# 18wA — Coherent contract-event probability mapping

Each selected predictive distribution is represented by the frozen 99 HKO-temperature quantiles. They are treated as equally weighted particles, so a contract probability is the fraction of particles belonging to that certified event. This gives non-negative probabilities that sum exactly to one within every date, decision rule and candidate.

Development probabilities are joined to development outcomes. The internal-holdout and June panel remains outcome blind and contains no market field.

**Revision v2.** Contract definitions are read from an installed outcome-free panel. Development outcomes are derived only from the already labelled 18vC OOF rows. The full 18s outcome panel is not opened.

In [1]:
from __future__ import annotations
import hashlib, json, math, platform, sys
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT=Path.cwd().resolve()
if not (ROOT/'.git').exists(): raise RuntimeError(f'Run from repository root, not {ROOT}')
UTC=timezone.utc; STEP='18wA'; PARTICLES=99; NCONTRACTS=11; EPS=1e-12
DEF=ROOT/'data/manual/18w_outcome_free_contract_definitions'
VA=ROOT/'data/processed/18vA_residual_model_design_and_features'
VC=ROOT/'data/processed/18vC_common_support_scoring_and_selection'
VD=ROOT/'data/processed/18vD_selected_models_blind_predictions'
CONTRACT_DEF=DEF/'18w_outcome_free_contract_definition_panel.csv'; DEF_META=DEF/'18w_outcome_free_contract_definition_metadata.json'
QGRID=VA/'18vA_quantile_grid.csv'; VA_SUM=VA/'18vA_summary.json'; VA_MAN=VA/'18vA_sha256_manifest.csv'
OOF=VC/'18vC_all_candidate_oof_predictions.csv'; COMMON=VC/'18vC_common_candidate_selection_support.csv'; SELECT=VC/'18vC_selected_candidates.json'; VC_SUM=VC/'18vC_summary.json'; VC_MAN=VC/'18vC_sha256_manifest.csv'
BLIND=VD/'18vD_selected_candidate_blind_predictions.csv'; REG=VD/'18vD_selected_candidate_registry.csv'; VD_SUM=VD/'18vD_summary.json'; VD_MAN=VD/'18vD_sha256_manifest.csv'
OUT=ROOT/'data/processed/18wA_contract_probability_mapping'; REPORT=ROOT/'reports/18wA_contract_probability_mapping'; OUT.mkdir(parents=True,exist_ok=True); REPORT.mkdir(parents=True,exist_ok=True)
for p in [CONTRACT_DEF,DEF_META,QGRID,VA_SUM,VA_MAN,OOF,COMMON,SELECT,VC_SUM,VC_MAN,BLIND,REG,VD_SUM,VD_MAN]:
    if not p.is_file(): raise FileNotFoundError(p)

def sha(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for c in iter(lambda:f.read(1024*1024),b''): h.update(c)
    return h.hexdigest()

def verify_manifest(p):
    m=pd.read_csv(p); bad=[]
    for r in m.itertuples(index=False):
        q=ROOT/r.path
        if not q.is_file(): bad.append('MISSING '+r.path); continue
        if sha(q)!=r.sha256: bad.append('HASH '+r.path)
        if q.stat().st_size!=int(r.size_bytes): bad.append('SIZE '+r.path)
    if bad: raise AssertionError('\n'.join(bad))

def pbool(s,name):
    if pd.api.types.is_bool_dtype(s): return s.astype(bool)
    x=s.astype(str).str.strip().str.lower().map({'true':True,'false':False,'1':True,'0':False,'yes':True,'no':False})
    if x.isna().any(): raise ValueError(f'Cannot parse {name}: {s[x.isna()].drop_duplicates().tolist()}')
    return x.astype(bool)

def order_book(g):
    z=g.copy(); z['_e']=z.event_type.map({'lower':0,'interior':1,'upper':2}); z['_l']=z.lower_bound_c.fillna(-1e9)
    z=z.sort_values(['_e','_l','upper_bound_c','market_id'],na_position='last').drop(columns=['_e','_l']).reset_index(drop=True)
    z['contract_order']=np.arange(len(z)); return z

def member(x,r):
    if r.event_type=='lower': return x<float(r.upper_bound_c)
    if r.event_type=='interior': return (x>=float(r.lower_bound_c))&(x<float(r.upper_bound_c))
    if r.event_type=='upper': return x>=float(r.lower_bound_c)
    raise ValueError(r.event_type)

def map_particles(x,book):
    x=np.asarray(x,float)
    if x.shape!=(PARTICLES,) or not np.isfinite(x).all(): raise AssertionError('Invalid 99-particle vector')
    b=order_book(book)
    if len(b)!=NCONTRACTS: raise AssertionError('Book does not have 11 contracts')
    M=np.column_stack([member(x,r) for r in b.itertuples(index=False)])
    if not np.all(M.sum(1)==1): raise AssertionError('Particle assigned to zero/multiple events')
    counts=M.sum(0).astype(int); probs=counts/PARTICLES
    if counts.sum()!=PARTICLES or not np.isclose(probs.sum(),1,atol=1e-12): raise AssertionError('Incoherent probabilities')
    b['particle_count']=counts; b['p_model_uncalibrated']=probs; return b

def date_weights(df):
    n=df.event_date.nunique(); w=1/(n*df.groupby('event_date').event_date.transform('size'))
    if not np.isclose(w.sum(),1,atol=1e-12): raise AssertionError('Weights')
    return w

for p in [VA_MAN,VC_MAN,VD_MAN]: verify_manifest(p)
for name,p in [('18vA',VA_SUM),('18vC',VC_SUM),('18vD',VD_SUM)]:
    s=json.loads(p.read_text())
    if s.get('verdict')!='PASS': raise AssertionError(f'{name} is not PASS')
def_meta=json.loads(DEF_META.read_text())
if def_meta.get('output_sha256')!=sha(CONTRACT_DEF): raise AssertionError('Outcome-free definition hash differs')
if def_meta.get('forbidden_outcome_columns_present') is not False: raise AssertionError('Definition metadata boundary failed')
selection=json.loads(SELECT.read_text())
selected=[selection['selected_baseline'],selection['selected_gaussian_process'],selection['selected_tree']]
if len(set(selected))!=3: raise AssertionError('Selected family representatives are not distinct')
contracts=pd.read_csv(CONTRACT_DEF,dtype={'market_id':str},low_memory=False); qgrid=pd.read_csv(QGRID); oof=pd.read_csv(OOF,low_memory=False); common=pd.read_csv(COMMON); blind=pd.read_csv(BLIND,low_memory=False); registry=pd.read_csv(REG)
for df in [contracts,oof,common,blind]: df['event_date']=pd.to_datetime(df.event_date,errors='raise')
oof['common_candidate_selection_support']=pbool(oof.common_candidate_selection_support,'oof common flag'); blind['outcome_blind']=pbool(blind.outcome_blind,'blind flag')
hq=qgrid.hko_quantile_column.tolist(); rq=qgrid.residual_quantile_column.tolist()
if len(hq)!=99 or len(rq)!=99: raise AssertionError('Quantile grid is not 99 points')
if len(contracts)!=1133 or contracts.event_date.nunique()!=103: raise AssertionError('Contract totals differ')
if not contracts.groupby('event_date').size().eq(11).all(): raise AssertionError('Invalid contract books')
forbidden_definition={'hko_daily_max_c','Y_event_int','Y_no_int','residual_c','forecast_error_c','p_market','market_binary_brier','market_binary_log_score'}
if forbidden_definition.intersection(contracts.columns): raise AssertionError('Outcome-free definition panel contains forbidden fields')
books={d:order_book(g) for d,g in contracts.groupby('event_date',sort=True)}
validation=[]
for d,b in books.items():
    lo=b[b.event_type.eq('lower')]; mid=b[b.event_type.eq('interior')].sort_values('lower_bound_c'); hi=b[b.event_type.eq('upper')]
    if (len(lo),len(mid),len(hi))!=(1,9,1): raise AssertionError(f'Event-type counts {d}')
    L=mid.lower_bound_c.to_numpy(float); U=mid.upper_bound_c.to_numpy(float)
    ok=np.allclose(U-L,1,atol=1e-12) and np.allclose(L[1:],U[:-1],atol=1e-12) and np.isclose(float(lo.upper_bound_c.iloc[0]),L[0]) and np.isclose(float(hi.lower_bound_c.iloc[0]),U[-1])
    if not ok: raise AssertionError(f'Non-contiguous book {d}')
    validation.append({'event_date':d,'contract_rows':11,'lower_contracts':1,'interior_contracts':9,'upper_contracts':1,'lower_endpoint_c':float(lo.upper_bound_c.iloc[0]),'upper_endpoint_c':float(hi.lower_bound_c.iloc[0]),'contiguous_exhaustive_partition':True})
validation=pd.DataFrame(validation)

devpred=oof[oof.candidate_id.isin(selected)&oof.common_candidate_selection_support].copy()
if len(devpred)!=408 or not devpred.groupby('candidate_id').size().eq(136).all(): raise AssertionError('Development prediction support differs')
devparts=[]
for r in devpred.itertuples(index=False):
    z=map_particles(np.array([getattr(r,c) for c in hq],float),books[r.event_date])
    realised=float(r.hko_daily_max_c)
    z['hko_daily_max_c']=realised
    z['Y_event_int']=[int(member(np.array([realised]),event)[0]) for event in z.itertuples(index=False)]
    z['Y_no_int']=1-z['Y_event_int']
    if int(z.Y_event_int.sum())!=1: raise AssertionError('Derived development outcome is not unique')
    for c,v in {'candidate_id':r.candidate_id,'model_family':r.model_family,'scope_type':r.scope_type,'scope_id':r.scope_id,'complexity_rank':r.complexity_rank,'decision_rule':r.decision_rule,'decision_rule_order':r.decision_rule_order,'development_fold':r.development_fold,'forecast_daily_max_c':r.forecast_daily_max_c,'predicted_residual_mean_c':r.predicted_residual_mean_c,'predicted_residual_median_c':r.predicted_residual_median_c,'mapping_distribution':'99_EQUAL_WEIGHT_HKO_QUANTILE_PARTICLES','calibration_variant':'UNCALIBRATED','date_balanced_selection_weight':r.date_balanced_selection_weight}.items(): z[c]=v
    devparts.append(z)
devprob=pd.concat(devparts,ignore_index=True)
if len(devprob)!=4488 or not devprob.groupby(['candidate_id','event_date','decision_rule']).p_model_uncalibrated.sum().apply(lambda v:np.isclose(v,1,atol=1e-12)).all(): raise AssertionError('Development probability panel')
bookrows=[]
for k,b in devprob.groupby(['candidate_id','event_date','decision_rule'],sort=True):
    w=b[b.Y_event_int.eq(1)]
    if len(w)!=1: raise AssertionError('Winner count')
    p=float(w.p_model_uncalibrated.iloc[0]); pv=b.p_model_uncalibrated.to_numpy(float); y=b.Y_event_int.to_numpy(float)
    bookrows.append({'candidate_id':k[0],'event_date':k[1],'decision_rule':k[2],'decision_rule_order':int(b.decision_rule_order.iloc[0]),'development_fold':int(b.development_fold.iloc[0]),'winning_market_id':str(w.market_id.iloc[0]),'winning_label':str(w.canonical_label.iloc[0]),'winning_probability':p,'categorical_log_score':-math.log(min(max(p,EPS),1.0)),'multiclass_brier':float(np.square(pv-y).sum()),'zero_winning_probability':p==0,'date_balanced_selection_weight':float(b.date_balanced_selection_weight.iloc[0])})
devscores=pd.DataFrame(bookrows).sort_values(['candidate_id','event_date','decision_rule_order']).reset_index(drop=True)
if len(devscores)!=408: raise AssertionError('Development book scores')

forbidden={'hko_daily_max_c','residual_c','forecast_error_c','Y_event_int','Y_no_int','p_market','market_binary_brier','market_binary_log_score','current_label_available_utc'}
if forbidden.intersection(blind.columns) or not blind.outcome_blind.all() or len(blind)!=477: raise AssertionError('Blind input boundary')
blindparts=[]
public=['event_date','market_id','condition_id','event_id','event_slug','market_slug','question','canonical_label','event_type','contract_event_type','lower_bound_c','upper_bound_c','bound_reference_c','selected_yes_token_id','no_token_id','sample_block','contract_order','particle_count','p_model_uncalibrated']
for r in blind.itertuples(index=False):
    z=map_particles(np.array([getattr(r,c) for c in hq],float),books[r.event_date]); z=z[[c for c in public if c in z.columns]].copy()
    for c,v in {'candidate_id':r.candidate_id,'model_family':r.model_family,'scope_type':r.scope_type,'scope_id':r.scope_id,'selection_roles':r.selection_roles,'complexity_rank':r.complexity_rank,'decision_rule':r.decision_rule,'decision_rule_order':r.decision_rule_order,'decision_cutoff_utc':r.decision_cutoff_utc,'evaluation_block':r.evaluation_block,'evaluation_stage':r.evaluation_stage,'forecast_daily_max_c':r.forecast_daily_max_c,'predicted_residual_mean_c':r.predicted_residual_mean_c,'predicted_residual_median_c':r.predicted_residual_median_c,'mapping_distribution':'99_EQUAL_WEIGHT_HKO_QUANTILE_PARTICLES','calibration_variant':'UNCALIBRATED','outcome_blind':True,'refit_on_holdout_labels':r.refit_on_holdout_labels,'fitted_parameter_source':r.fitted_parameter_source}.items(): z[c]=v
    blindparts.append(z)
blindprob=pd.concat(blindparts,ignore_index=True)
if len(blindprob)!=5247 or not blindprob.groupby(['candidate_id','event_date','decision_rule']).p_model_uncalibrated.sum().apply(lambda v:np.isclose(v,1,atol=1e-12)).all(): raise AssertionError('Blind probability panel')
if forbidden.intersection(blindprob.columns) or pbool(blindprob.refit_on_holdout_labels,'refit').any(): raise AssertionError('Blind output boundary')

summaryrows=[]
for cid,g in devscores.groupby('candidate_id'):
    w=g.date_balanced_selection_weight.to_numpy(float)
    summaryrows.append({'candidate_id':cid,'development_books':len(g),'development_dates':g.event_date.nunique(),'date_balanced_mean_categorical_log_score':float(np.average(g.categorical_log_score,weights=w)),'date_balanced_mean_multiclass_brier':float(np.average(g.multiclass_brier,weights=w)),'zero_winning_probability_books':int(g.zero_winning_probability.sum())})
score_summary=pd.DataFrame(summaryrows)
checks=pd.DataFrame([
{'check':'contract_dates_103','passed':len(validation)==103,'detail':f'dates={len(validation)}','blocking':True},
{'check':'development_probability_rows_4488','passed':len(devprob)==4488,'detail':f'rows={len(devprob)}','blocking':True},
{'check':'development_book_scores_408','passed':len(devscores)==408,'detail':f'rows={len(devscores)}','blocking':True},
{'check':'blind_probability_rows_5247','passed':len(blindprob)==5247,'detail':f'rows={len(blindprob)}','blocking':True},
{'check':'blind_outcomes_absent','passed':not bool(forbidden.intersection(blindprob.columns)),'detail':'outcome-blind release','blocking':True},
{'check':'full_outcome_panel_not_loaded','passed':True,'detail':'18wA reads the installed outcome-free contract-definition panel only','blocking':True},
{'check':'market_information_absent','passed':'p_market' not in blindprob.columns,'detail':'weather model only','blocking':True},
{'check':'probability_bridge_absent','passed':not any('gaussian_bridge' in c.lower() or 'ecmwf_proxy' in c.lower() for c in blindprob.columns),'detail':'no bridge field','blocking':True},
])
if not checks.passed.all(): raise AssertionError(checks[~checks.passed].to_string(index=False))
issues=pd.DataFrame(columns=['issue_level','issue_code','candidate_id','event_date','decision_rule','market_id','detail','blocking'])

outputs={
'18wA_contract_book_validation.csv':validation,
'18wA_development_uncalibrated_probability_panel.csv':devprob,
'18wA_development_uncalibrated_book_scores.csv':devscores,
'18wA_development_uncalibrated_score_summary.csv':score_summary,
'18wA_blind_uncalibrated_probability_panel.csv':blindprob,
'18wA_integrity_checks.csv':checks,
'18wA_issues.csv':issues,
}
for name,df in outputs.items():
    z=df.copy()
    for c in z.columns:
        if 'date' in c.lower() and pd.api.types.is_datetime64_any_dtype(z[c]): z[c]=z[c].dt.strftime('%Y-%m-%d')
        if 'cutoff' in c.lower() or c.lower().endswith('_utc') or 'available' in c.lower() or 'initialisation' in c.lower(): z[c]=z[c].astype('string')
    z.to_csv(OUT/name,index=False)
protocol={'step':STEP,'generated_at_utc':datetime.now(UTC).isoformat(),'verdict':'PASS','mapping_distribution':'equal-weight empirical distribution on 99 HKO predictive quantiles','particle_mass':1/PARTICLES,'contract_event_sets':{'lower':'(-infinity, upper_bound_c)','interior':'[lower_bound_c, upper_bound_c)','upper':'[lower_bound_c, infinity)'},'selected_candidates':selected,'development_common_books_per_candidate':136,'blind_books_per_candidate':159,'probabilities_sum_to_one':True,'contract_definition_source':'installed outcome-free definition panel','development_outcomes_source':'18vC development OOF labels','full_outcome_panel_loaded':False,'blind_outcomes_loaded':False,'market_information_used':False,'probability_bridge_retained':False,'calibration_pending':True}
(OUT/'18wA_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
sources=pd.DataFrame([
{'input_role':'outcome_free_contract_definition_panel','path':str(CONTRACT_DEF.relative_to(ROOT)),'rows':len(contracts),'sha256':sha(CONTRACT_DEF)},
{'input_role':'outcome_free_contract_definition_metadata','path':str(DEF_META.relative_to(ROOT)),'rows':1,'sha256':sha(DEF_META)},
{'input_role':'18vA_quantile_grid','path':str(QGRID.relative_to(ROOT)),'rows':len(qgrid),'sha256':sha(QGRID)},
{'input_role':'18vC_all_candidate_oof','path':str(OOF.relative_to(ROOT)),'rows':len(oof),'sha256':sha(OOF)},
{'input_role':'18vC_common_selection_keys','path':str(COMMON.relative_to(ROOT)),'rows':len(common),'sha256':sha(COMMON)},
{'input_role':'18vC_selection','path':str(SELECT.relative_to(ROOT)),'rows':1,'sha256':sha(SELECT)},
{'input_role':'18vD_blind_predictions','path':str(BLIND.relative_to(ROOT)),'rows':len(blind),'sha256':sha(BLIND)},
]); sources.to_csv(OUT/'18wA_source_inventory.csv',index=False)
summary={'step':STEP,'generated_at_utc':datetime.now(UTC).isoformat(),'verdict':'PASS','contract_dates':103,'contracts':1133,'selected_candidates':3,'predictive_particles_per_book':99,'development_books_per_candidate':136,'development_prediction_rows':408,'development_contract_probability_rows':4488,'blind_books_per_candidate':159,'blind_prediction_rows':477,'blind_contract_probability_rows':5247,'full_outcome_panel_loaded':False,'market_information_used':False,'blind_outcomes_loaded':False,'probability_bridge_retained':False,'calibration_pending':True,'issue_rows':0,'integrity_checks_passed':int(checks.passed.sum()),'integrity_checks_total':len(checks)}
(OUT/'18wA_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
(OUT/'18wA_environment.json').write_text(json.dumps({'generated_at_utc':datetime.now(UTC).isoformat(),'python':sys.version,'platform':platform.platform(),'pandas':pd.__version__,'numpy':np.__version__,'revision':'v2'},indent=2),encoding='utf-8')
lines=['# 18wA coherent contract-event probability mapping','','**PASS**','','Each predictive distribution is mapped using 99 equally weighted HKO quantile particles. Every particle belongs to exactly one certified event and every model book sums to one.','','## Development uncalibrated scores','','| Candidate | Books | Categorical log | Multiclass Brier | Zero winner probabilities |','|---|---:|---:|---:|---:|']
for r in score_summary.itertuples(index=False): lines.append(f'| {r.candidate_id} | {int(r.development_books)} | {r.date_balanced_mean_categorical_log_score:.6f} | {r.date_balanced_mean_multiclass_brier:.6f} | {int(r.zero_winning_probability_books)} |')
lines += ['','The blind probability panel contains no realised outcome or market variable.']
(REPORT/'18wA_contract_probability_mapping_report.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
manifest=[]
for root in [OUT,REPORT]:
    for p in sorted(root.rglob('*')):
        if p.is_file() and p.name!='18wA_sha256_manifest.csv': manifest.append({'path':str(p.relative_to(ROOT)),'size_bytes':p.stat().st_size,'sha256':sha(p)})
pd.DataFrame(manifest).to_csv(OUT/'18wA_sha256_manifest.csv',index=False)
print(json.dumps(summary,indent=2)); display(score_summary); print('18wA contract-probability mapping release: PASS')

{
  "step": "18wA",
  "generated_at_utc": "2026-07-22T12:26:58.181263+00:00",
  "verdict": "PASS",
  "contract_dates": 103,
  "contracts": 1133,
  "selected_candidates": 3,
  "predictive_particles_per_book": 99,
  "development_books_per_candidate": 136,
  "development_prediction_rows": 408,
  "development_contract_probability_rows": 4488,
  "blind_books_per_candidate": 159,
  "blind_prediction_rows": 477,
  "blind_contract_probability_rows": 5247,
  "full_outcome_panel_loaded": false,
  "market_information_used": false,
  "blind_outcomes_loaded": false,
  "probability_bridge_retained": false,
  "calibration_pending": true,
  "issue_rows": 0,
  "integrity_checks_passed": 8,
  "integrity_checks_total": 8
}


,candidate_id,development_books,development_dates,date_balanced_mean_categorical_log_score,date_balanced_mean_multiclass_brier,zero_winning_probability_books
0,catboost_quantile_pooled,136,36,9.017854,0.829272,44
1,gp_matern32_rule,136,36,2.292783,0.777329,4
2,pooled_empirical_residual,136,36,2.238408,0.751244,4


18wA contract-probability mapping release: PASS
